# Selección del Mejor Modelo - Diabetes

### By:
Maria Camila Aristizábal Aguirre

### Date:
2026-08-19

## 📚 Import  libraries

In [2]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## 💾 Load data

In [3]:
MODEL_INPUT_DIR = Path("../../data/05_model_input")
train = pd.read_parquet(MODEL_INPUT_DIR / "train.parquet")
test = pd.read_parquet(MODEL_INPUT_DIR / "test.parquet")

COLUMNAS_FEATURES = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]
TARGET = "Outcome"

X_train, y_train = train[COLUMNAS_FEATURES], train[TARGET].astype(bool)
X_test, y_test = test[COLUMNAS_FEATURES], test[TARGET].astype(bool)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (629, 8) | Test: (158, 8)


## 👷 Data preparation

##  Decisiones

- Se reutiliza el split de `data/05_model_input/` ya imputado — no se
  genera uno nuevo.
- Se comparan 5 modelos de familias distintas: Regresión Logística, Árbol de
  Decisión, Random Forest, KNN y SVM.
- Se aplica `StandardScaler` de forma uniforme en el pipeline de los 5 modelos:
  no afecta a los árboles (invariantes a escalado monótono) y es necesario para
  Regresión Logística, KNN y SVM.
- Comparación inicial con `KFold(5)`; se seleccionan los 2 mejores modelos por F1
  para afinar con `GridSearchCV(cv=5)`, incluyendo `class_weight="balanced"` como
  hiperparámetro a probar (dataset con desbalance moderado 65/35).
- No se aplica prueba ANOVA ni curvas de aprendizaje (`ShuffleSplit`): para el
  alcance de esta tarea basta reportar media ± desviación estándar en CV.
- Métrica principal de selección: **F1** (balancea precision y recall, a raíz de lo
  aprendido en la Tarea 5 sobre el trade-off entre ambas).

## Modelos candidatos (pipeline con escalado uniforme)

In [4]:
SEMILLA = 42

modelos_candidatos = {
    "Regresion Logistica": LogisticRegression(random_state=SEMILLA, max_iter=1000),
    "Arbol de Decision": DecisionTreeClassifier(random_state=SEMILLA),
    "Random Forest": RandomForestClassifier(random_state=SEMILLA),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(random_state=SEMILLA),
}

pipelines = {
    nombre: Pipeline(steps=[("scaler", StandardScaler()), ("model", modelo)])
    for nombre, modelo in modelos_candidatos.items()
}

## Comparación inicial con KFold(5)

In [5]:
K_FOLDS = 5
METRICAS = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEMILLA)

filas_comparacion = []
for nombre, pipe in pipelines.items():
    resultados = cross_validate(pipe, X_train, y_train, cv=cv, scoring=METRICAS)
    fila = {"modelo": nombre}
    for metrica in METRICAS:
        fila[f"{metrica}_media"] = resultados[f"test_{metrica}"].mean()
        fila[f"{metrica}_std"] = resultados[f"test_{metrica}"].std()
    filas_comparacion.append(fila)

tabla_comparacion = pd.DataFrame(filas_comparacion).set_index("modelo")
tabla_comparacion.sort_values("f1_media", ascending=False)

,accuracy_media,accuracy_std,precision_media,precision_std,recall_media,recall_std,f1_media,f1_std,roc_auc_media,roc_auc_std
modelo,,,,,,,,,,
Random Forest,0.772660,0.028165,0.698989,0.035298,0.618452,0.077624,0.654670,0.055698,0.835152,0.023786
KNN,0.747238,0.017382,0.649541,0.050848,0.617152,0.054884,0.631033,0.039003,0.795574,0.023730
Regresion Logistica,0.759873,0.049887,0.705630,0.063052,0.551558,0.069245,0.618738,0.066852,0.819872,0.043427
SVM,0.747175,0.039309,0.683707,0.068851,0.515956,0.080946,0.587409,0.077055,0.823746,0.035173
Arbol de Decision,0.683683,0.022497,0.544823,0.061574,0.600078,0.066289,0.570074,0.057789,0.663896,0.032696


## selección por promedio (3 modelos) + tuning + prueba estadística

In [ ]:
promedio_f1 = tabla_comparacion["f1_media"].mean()
modelos_seleccionados = tabla_comparacion[tabla_comparacion["f1_media"] > promedio_f1].sort_values(
    "f1_media", ascending=False
)
nombres_seleccionados = modelos_seleccionados.index.tolist()

print(f"Promedio de F1 entre los 5 modelos: {promedio_f1:.4f}")
print(f"Modelos por encima del promedio (se afinan): {nombres_seleccionados}")
modelos_seleccionados[["f1_media", "f1_std", "recall_media", "precision_media"]]

Promedio de F1 entre los 5 modelos: 0.6124
Modelos por encima del promedio (se afinan): ['Random Forest', 'KNN', 'Regresion Logistica']


,f1_media,f1_std,recall_media,precision_media
modelo,,,,
Random Forest,0.654670,0.055698,0.618452,0.698989
KNN,0.631033,0.039003,0.617152,0.649541
Regresion Logistica,0.618738,0.066852,0.551558,0.705630


## Afinamiento de hiperparámetros (GridSearchCV, cv=5)

In [13]:
param_grids = {
    "Random Forest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 5, 10, 15],
        "model__min_samples_leaf": [1, 5, 10],
        "model__class_weight": [None, "balanced"],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 15],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],
    },
    "Regresion Logistica": {
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__class_weight": [None, "balanced"],
        "model__solver": ["liblinear", "lbfgs"],
    },
}

In [14]:
modelos_afinados = {}
for nombre in nombres_seleccionados:
    grid = GridSearchCV(
        pipelines[nombre],
        param_grid=param_grids[nombre],
        cv=cv,
        scoring="f1",
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    modelos_afinados[nombre] = grid
    print(f"--- {nombre} ---")
    print(f"Mejor F1 (CV): {grid.best_score_:.4f}")
    print(f"Mejores hiperparametros: {grid.best_params_}")
    print()

--- Random Forest ---
Mejor F1 (CV): 0.6996
Mejores hiperparametros: {'model__class_weight': 'balanced', 'model__max_depth': 5, 'model__min_samples_leaf': 10, 'model__n_estimators': 200}

--- KNN ---
Mejor F1 (CV): 0.6312
Mejores hiperparametros: {'model__n_neighbors': 5, 'model__p': 2, 'model__weights': 'distance'}

--- Regresion Logistica ---
Mejor F1 (CV): 0.6674
Mejores hiperparametros: {'model__C': 0.01, 'model__class_weight': 'balanced', 'model__solver': 'liblinear'}



## Prueba estadística entre los modelos afinados

In [15]:
from itertools import combinations

import numpy as np
from scipy import stats
from sklearn.model_selection import cross_val_score

scores_f1_por_modelo = {
    nombre: cross_val_score(grid.best_estimator_, X_train, y_train, cv=cv, scoring="f1")
    for nombre, grid in modelos_afinados.items()
}

for nombre, scores in scores_f1_por_modelo.items():
    print(f"{nombre}: F1 por fold = {np.round(scores, 3)}")

print("\nComparaciones pareadas (Wilcoxon signed-rank sobre F1 por fold):")
for modelo_a, modelo_b in combinations(scores_f1_por_modelo.keys(), 2):
    estadistico, p_valor = stats.wilcoxon(
        scores_f1_por_modelo[modelo_a], scores_f1_por_modelo[modelo_b]
    )
    print(f"{modelo_a} vs {modelo_b}: p-valor = {p_valor:.4f}")

Random Forest: F1 por fold = [0.681 0.622 0.768 0.698 0.729]
KNN: F1 por fold = [0.557 0.634 0.674 0.632 0.659]
Regresion Logistica: F1 por fold = [0.63  0.632 0.792 0.647 0.636]

Comparaciones pareadas (Wilcoxon signed-rank sobre F1 por fold):
Random Forest vs KNN: p-valor = 0.1250
Random Forest vs Regresion Logistica: p-valor = 0.3125
KNN vs Regresion Logistica: p-valor = 0.4375


## Ajuste del umbral de decisión

In [18]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import cross_val_predict

resultados_umbral_ajustado = []
umbrales_elegidos = {}

for nombre, grid in modelos_afinados.items():
    modelo = grid.best_estimator_
    proba_oof = cross_val_predict(modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    precisiones, recalls, umbrales = precision_recall_curve(y_train, proba_oof)
    f1_por_umbral = 2 * (precisiones * recalls) / (precisiones + recalls + 1e-9)
    mejor_indice = f1_por_umbral[:-1].argmax()
    mejor_umbral = umbrales[mejor_indice]
    umbrales_elegidos[nombre] = mejor_umbral

    proba_test = modelo.predict_proba(X_test)[:, 1]
    y_pred_test = (proba_test >= mejor_umbral).astype(int)

    resultados_umbral_ajustado.append(
        {
            "modelo": f"{nombre} (umbral={mejor_umbral:.3f})",
            "accuracy": accuracy_score(y_test, y_pred_test),
            "precision": precision_score(y_test, y_pred_test),
            "recall": recall_score(y_test, y_pred_test),
            "f1": f1_score(y_test, y_pred_test),
        }
    )

tabla_umbral_ajustado = pd.DataFrame(resultados_umbral_ajustado).set_index("modelo")
tabla_umbral_ajustado.sort_values("f1", ascending=False)

,accuracy,precision,recall,f1
modelo,,,,
Random Forest (umbral=0.436),0.740506,0.592593,0.857143,0.700730
KNN (umbral=0.358),0.759494,0.636364,0.750000,0.688525
Regresion Logistica (umbral=0.449),0.721519,0.575000,0.821429,0.676471


## Ensamble de los 2 mejores modelos afinados

In [19]:
from sklearn.ensemble import VotingClassifier

ensamble = VotingClassifier(
    estimators=[
        ("random_forest", modelos_afinados["Random Forest"].best_estimator_),
        (
            "regresion_logistica",
            modelos_afinados["Regresion Logistica"].best_estimator_,
        ),
    ],
    voting="soft",
)
ensamble.fit(X_train, y_train)

proba_oof_ensamble = cross_val_predict(ensamble, X_train, y_train, cv=cv, method="predict_proba")[
    :, 1
]
precisiones, recalls, umbrales = precision_recall_curve(y_train, proba_oof_ensamble)
f1_por_umbral = 2 * (precisiones * recalls) / (precisiones + recalls + 1e-9)
mejor_indice = f1_por_umbral[:-1].argmax()
mejor_umbral_ensamble = umbrales[mejor_indice]

proba_test_ensamble = ensamble.predict_proba(X_test)[:, 1]
y_pred_ensamble = (proba_test_ensamble >= mejor_umbral_ensamble).astype(int)

print(f"Umbral elegido (ensamble, via OOF en train): {mejor_umbral_ensamble:.3f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_ensamble):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_ensamble):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_ensamble):.4f}")
print(f"F1: {f1_score(y_test, y_pred_ensamble):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, proba_test_ensamble):.4f}")

Umbral elegido (ensamble, via OOF en train): 0.497
Accuracy: 0.7342
Precision: 0.6000
Recall: 0.7500
F1: 0.6667
ROC-AUC: 0.8447


## Evaluación final en test (finalistas afinados vs. baseline de la Tarea 5)

In [10]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def evaluar_en_test(modelo, nombre, X_test, y_test):
    y_pred = modelo.predict(X_test)
    fila = {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    }
    if hasattr(modelo, "predict_proba"):
        y_score = modelo.predict_proba(X_test)[:, 1]
        fila["roc_auc"] = roc_auc_score(y_test, y_score)
    elif hasattr(modelo, "decision_function"):
        y_score = modelo.decision_function(X_test)
        fila["roc_auc"] = roc_auc_score(y_test, y_score)
    else:
        fila["roc_auc"] = None
    return fila


resultados_test = [
    evaluar_en_test(grid.best_estimator_, f"{nombre} (afinado)", X_test, y_test)
    for nombre, grid in modelos_afinados.items()
]

# Regla heuristica de la Tarea 5 (Glucose > 127) recreada directamente:
# no se puede des-serializar la clase original porque se definio en el
# namespace de otro notebook (limitacion normal de joblib/pickle con
# clases definidas ad-hoc, no un error de este notebook).
UMBRAL_GLUCOSE = 127
y_pred_heuristico = (X_test["Glucose"] > UMBRAL_GLUCOSE).astype(int)

resultados_test.append(
    {
        "modelo": "Heuristico Glucose (Tarea 5)",
        "accuracy": accuracy_score(y_test, y_pred_heuristico),
        "precision": precision_score(y_test, y_pred_heuristico),
        "recall": recall_score(y_test, y_pred_heuristico),
        "f1": f1_score(y_test, y_pred_heuristico),
        "roc_auc": None,
    }
)

tabla_test = pd.DataFrame(resultados_test).set_index("modelo")
tabla_test.sort_values("f1", ascending=False)

,accuracy,precision,recall,f1,roc_auc
modelo,,,,,
Heuristico Glucose (Tarea 5),0.797468,0.730769,0.678571,0.703704,NaN
Random Forest (afinado),0.734177,0.606061,0.714286,0.655738,0.832108
KNN (afinado),0.746835,0.700000,0.500000,0.583333,0.793855


## Guardado del modelo final y tablas de comparación

In [ ]:
MODELS_DIR = Path("../../data/06_models")
modelo_final_random_forest = modelos_afinados["Random Forest"].best_estimator_
joblib.dump(modelo_final_random_forest, MODELS_DIR / "modelo_final_random_forest.joblib")

MODEL_OUTPUT_DIR = Path("../../data/07_model_output")
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tabla_comparacion.to_csv(MODEL_OUTPUT_DIR / "seleccion_modelos_cv.csv")
tabla_test.to_csv(MODEL_OUTPUT_DIR / "seleccion_modelos_test.csv")

## Hallazgos - Selección del Mejor Modelo

### Comparación inicial (CV, KFold=5, train)
| Modelo | F1 media | F1 std | ROC-AUC media |
|---|---|---|---|
| Random Forest | 0.655 | 0.056 | 0.835 |
| KNN | 0.631 | 0.039 | 0.796 |
| Regresión Logística | 0.619 | 0.067 | 0.820 |
| SVM | 0.587 | 0.077 | 0.824 |
| Árbol de Decisión | 0.570 | 0.058 | 0.664 |

Random Forest y KNN quedaron como finalistas. Cabe notar que KNN superó a Regresión
Logística y SVM, contrario a lo esperado dado que es más sensible al desbalance de
clases y a outliers remanentes que los otros modelos.

### Afinamiento de hiperparámetros (GridSearchCV, cv=5, scoring=F1)
- **Random Forest**: mejores hiperparámetros
  `{class_weight: "balanced", max_depth: 5, min_samples_leaf: 10, n_estimators: 200}`,
  F1 en CV = 0.6996 (mejora sobre el 0.655 sin afinar).
- **KNN**: mejores hiperparámetros `{n_neighbors: 5, p: 2, weights: "distance"}`,
  F1 en CV = 0.6312 (mejora marginal sobre el 0.631 sin afinar).

### Evaluación final en test — hallazgo principal
| Modelo | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| Heurístico Glucose (Tarea 5) | 0.797 | 0.731 | 0.679 | **0.704** | — |
| Random Forest (afinado) | 0.734 | 0.606 | 0.714 | 0.656 | 0.832 |
| KNN (afinado) | 0.747 | 0.700 | 0.500 | 0.583 | 0.794 |

**El heurístico de la Tarea 5 supera en F1 a los dos modelos afinados en el
conjunto de test**, a pesar de que Random Forest tuvo mejor F1 en CV (0.6996 vs.
0.656 en test). La causa más probable no es que el heurístico sea "mejor modelo",
sino tres factores metodológicos:
1. `GridSearchCV` evaluó 72 combinaciones de hiperparámetros para Random Forest y
   se quedó con la de mejor score de CV — con tantos intentos, es esperable que la
   combinación ganadora tenga un score de CV optimista solo por azar (sesgo de
   selección / "winner's curse"), lo cual explica la brecha entre el F1 de CV
   (0.6996) y el de test (0.6558).
2. El dataset es pequeño (629 train, 158 test), lo que da alta varianza tanto al CV
   como a la estimación en un único split de test.
3. El heurístico es una regla fija que no se ajusta a los datos, por lo que no
   tiene el riesgo de sobreajuste al proceso de búsqueda de hiperparámetros que sí
   tienen los modelos afinados.

### Dos decisiones separadas, cada una con su propia justificación
1. **Por la métrica definida (F1) en el conjunto de test, el heurístico de la Tarea
   5 es el mejor modelo de este ejercicio de comparación.** Esta es la conclusión:
   no siempre un modelo más complejo supera a una regla simple, y
   aquí no lo hizo.
2. **Para la Tarea 7 (Interpretación del Modelo) se lleva Random Forest (afinado),
   no por desempeño, sino porque el objetivo pedagógico de esa tarea (interpretar
   importancia de variables en un modelo entrenado) requiere un modelo con más de
   una variable de decisión — el heurístico de una sola variable ya se interpretó
   por completo en la Tarea 5.** Random Forest también ofrece una ventaja práctica
   real que el heurístico no tiene: un score de probabilidad continuo (ROC-AUC =
   0.832) que permite ajustar el umbral de decisión según el contexto clínico,
   algo que una regla binaria dura no permite sin volverse una regla más compleja.